In [1]:
# Assignment #4: Text Generation (Seq2Seq) Performance Analysis
# Task: Compare Google vs Facebook/Meta translator on Eng→Ur dataset using BLEU + ROUGE.

print("Assignment #4: Seq2Seq Performance Analysis (Eng→Ur)")
print("Metrics: BLEU + ROUGE, Model outputs: Google Translator, Meta (M2M100)")

Assignment #4: Seq2Seq Performance Analysis (Eng→Ur)
Metrics: BLEU + ROUGE, Model outputs: Google Translator, Meta (M2M100)


**Install dependencies**

In [2]:
!pip -q install deep-translator sacrebleu evaluate transformers sentencepiece accelerate pandas
print("Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
Dependencies installed


**Imports**

In [3]:
import os, json, zipfile
from typing import List, Optional, Tuple, Dict

import pandas as pd
import torch

from deep_translator import GoogleTranslator
import sacrebleu
import evaluate

from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

**Upload ZIP in Colab**

In [7]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

print("Accessing: Urdu_Eng_Dataset.zip from Google Drive")

# Set path to your ZIP file
zip_path = "/content/drive/MyDrive/Datasets/Urdu_Eng_Dataset.zip"

# Check if file exists
if not os.path.exists(zip_path):
    raise FileNotFoundError("ZIP file not found in Drive. Please check the path.")

print("File found:", zip_path)

Mounted at /content/drive
Accessing: Urdu_Eng_Dataset.zip from Google Drive
File found: /content/drive/MyDrive/Datasets/Urdu_Eng_Dataset.zip


**Unzip dataset**

In [8]:
extract_dir = "Urdu_Eng_Dataset"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Unzipped to:", extract_dir)

# Show extracted files
all_files = []
for root, _, files_ in os.walk(extract_dir):
    for fn in files_:
        all_files.append(os.path.join(root, fn))

print("Files found:")
for f in all_files[:50]:
    print(" -", f)
if len(all_files) > 50:
    print(f"... and {len(all_files)-50} more")

Unzipped to: Urdu_Eng_Dataset
Files found:
 - Urdu_Eng_Dataset/english-corpus.txt
 - Urdu_Eng_Dataset/urdu-corpus.txt
 - Urdu_Eng_Dataset/__MACOSX/._urdu-corpus.txt
 - Urdu_Eng_Dataset/__MACOSX/._english-corpus.txt


**Load dataset (auto-detect file)**

In [9]:
def pick_main_file(files_list: List[str]) -> str:
    supported = [f for f in files_list if f.lower().endswith((".csv", ".tsv", ".jsonl", ".json", ".txt"))]
    if not supported:
        raise FileNotFoundError(" No supported file found (csv/tsv/json/jsonl/txt).")
    supported.sort(key=lambda p: os.path.getsize(p), reverse=True)
    return supported[0]

data_path = pick_main_file(all_files)
print("Auto-selected dataset file:", data_path)

# Load based on extension
low = data_path.lower()
if low.endswith(".csv"):
    df_raw = pd.read_csv(data_path)
elif low.endswith(".tsv"):
    df_raw = pd.read_csv(data_path, sep="\t")
elif low.endswith(".jsonl"):
    rows = []
    with open(data_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    df_raw = pd.DataFrame(rows)
elif low.endswith(".json"):
    with open(data_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    df_raw = pd.DataFrame(obj if isinstance(obj, list) else obj.get("data", obj))
else:
    # txt fallback: try "English<TAB>Urdu"
    rows = []
    with open(data_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if "\t" in line:
                a, b = line.split("\t", 1)
                rows.append({"english": a.strip(), "urdu": b.strip()})
    df_raw = pd.DataFrame(rows)

print("Loaded shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
df_raw.head()

Auto-selected dataset file: Urdu_Eng_Dataset/urdu-corpus.txt
Loaded shape: (0, 0)
Columns: []


""


**Detect English + Urdu columns (clean)**

In [10]:
import os
import pandas as pd

#  Change only if your unzip folder name is different
DATA_DIR = "Urdu_Eng_Dataset"   # same folder you used in unzip cell

def find_file_by_keywords(root: str, keywords: list[str]) -> str:
    """
    Find a file inside root directory whose filename contains all keywords (case-insensitive).
    Example: keywords=["english", "corpus"] matches english-corpus.txt
    """
    keywords = [k.lower() for k in keywords]
    matches = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            name = fn.lower()
            if all(k in name for k in keywords):
                matches.append(os.path.join(dirpath, fn))
    if not matches:
        raise FileNotFoundError(f"Could not find file with keywords {keywords} inside {root}")
    # pick the largest match (usually main file)
    matches.sort(key=lambda p: os.path.getsize(p), reverse=True)
    return matches[0]

# Auto-find both files
en_path = find_file_by_keywords(DATA_DIR, ["english"])
ur_path = find_file_by_keywords(DATA_DIR, ["urdu"])

print("English file:", en_path)
print("Urdu file   :", ur_path)

def read_lines(path: str) -> list[str]:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = [line.strip() for line in f if line.strip()]
    return lines

english_lines = read_lines(en_path)
urdu_lines = read_lines(ur_path)

print("English lines:", len(english_lines))
print("Urdu lines   :", len(urdu_lines))

min_len = min(len(english_lines), len(urdu_lines))
if len(english_lines) != len(urdu_lines):
    print("Warning: line counts differ. Using first", min_len, "pairs only.")

df = pd.DataFrame({
    "english": english_lines[:min_len],
    "urdu_ref": urdu_lines[:min_len],
})

# Clean
df["english"] = df["english"].astype(str).str.strip()
df["urdu_ref"] = df["urdu_ref"].astype(str).str.strip()
df = df[(df["english"].str.len() > 0) & (df["urdu_ref"].str.len() > 0)].reset_index(drop=True)

print("Final usable rows:", len(df))
df.head(10)

English file: Urdu_Eng_Dataset/english-corpus.txt
Urdu file   : Urdu_Eng_Dataset/urdu-corpus.txt
English lines: 24525
Urdu lines   : 24524
Final usable rows: 24524


,english,urdu_ref
0,is zain your nephew,زین تمہارا بھتیجا ہے۔
1,i wish youd trust me,کاش تم مجھ پر بھروسہ کرتے
2,did he touch you,کیا اس نے آپ کو چھوا؟
3,its part of life,اس کی زندگی کا حصہ
4,zain isnt ugly,زین بدصورت نہیں ہے۔
5,above all be patient,سب سے بڑھ کر صبر کرو
6,i learned it from him,میں نے اسے اس سے سیکھا۔
7,why am i doing this,میں یہ کیوں کر رہا ہوں
8,i made a bad decision,میں نے ایک برا فیصلہ کیا
9,zain wont care,زین پرواہ نہیں کرے گا


**Google Translator setup + translate function**

In [11]:
from deep_translator import GoogleTranslator

# Google Translator object (EN -> UR)
google_translator = GoogleTranslator(source="en", target="ur")

def translate_google(text: str) -> str:
    """
    Translate one English sentence to Urdu using Google (deep-translator).
    If translation fails, return "" so pipeline continues.
    """
    text = str(text).strip()
    if not text:
        return ""
    try:
        return google_translator.translate(text) or ""
    except Exception:
        return ""

print("Google translator ready")
print("Demo:", translate_google("My name is Ali."))

Google translator ready
Demo: میرا نام علی ہے۔


**Facebook/Meta Translator (Seq2Seq) setup**

In [12]:
import torch
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# Device select (GPU  fast)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load tokenizer + model
tok = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")
model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M").to(device)
model.eval()

# Set source language English and force output language Urdu
tok.src_lang = "en"
forced_bos_token_id_ur = tok.get_lang_id("ur")

def translate_meta_batch(texts, max_new_tokens: int = 128, num_beams: int = 4):
    """
    Translate a batch of English sentences to Urdu using Meta M2M100.
    """
    texts = [str(t).strip() for t in texts]
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True).to(device)

    with torch.no_grad():
        out = model.generate(
            **enc,
            forced_bos_token_id=forced_bos_token_id_ur,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams
        )

    return [x.strip() for x in tok.batch_decode(out, skip_special_tokens=True)]

print("Meta (Facebook) translator ready")
print("Demo:", translate_meta_batch(["My name is Humera."])[0])

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

Meta (Facebook) translator ready
Demo: میرا نام ہومرا ہے


In [13]:
print("Rows:", len(df))
print("Columns:", list(df.columns))
print("Missing English:", df["english"].isna().sum())
print("Missing Urdu Ref:", df["urdu_ref"].isna().sum())

# quick sample for class
display(df.sample(5, random_state=42))

Rows: 24524
Columns: ['english', 'urdu_ref']
Missing English: 0
Missing Urdu Ref: 0


,english,urdu_ref
14443,it’s very thoughtful of you,کیا وہ بھائی ہیں؟
24319,zain has ambition,میں نے رسی کو چھوڑ دیا۔
12678,you may enter now,زین نے مجھے گلے لگایا
24423,my dad will kill me,وہ انگریزی پڑھاتی ہے۔
19977,i havent got all day,کیا میں آپ کو گانا چلا سکتا ہوں؟


**Run translations on full dataset (Google + Meta) and store in df**

In [14]:
import os
import pandas as pd

os.makedirs("outputs", exist_ok=True)

GOOGLE_FILE = "outputs/google.csv"
META_FILE   = "outputs/meta.csv"

# -------------------- Google (save + resume) --------------------
if os.path.exists(GOOGLE_FILE):
    google_preds = pd.read_csv(GOOGLE_FILE)["pred"].astype(str).tolist()
    print("Loaded Google saved rows:", len(google_preds))
else:
    google_preds = []
    print("No Google cache found, starting from 0")

total = len(df)

for i in range(len(google_preds), total):
    if i % 50 == 0:
        print(f"Google resume: {i}/{total}")
    google_preds.append(translate_google(df.loc[i, "english"]))

    # Save every 50 rows (safe)
    if (i + 1) % 50 == 0:
        pd.DataFrame({"pred": google_preds}).to_csv(GOOGLE_FILE, index=False, encoding="utf-8-sig")

# Final save
pd.DataFrame({"pred": google_preds}).to_csv(GOOGLE_FILE, index=False, encoding="utf-8-sig")
df["urdu_google"] = google_preds
print("Google done:", len(google_preds))


# -------------------- Meta (save + resume) --------------------
if os.path.exists(META_FILE):
    meta_preds = pd.read_csv(META_FILE)["pred"].astype(str).tolist()
    print("Loaded Meta saved rows:", len(meta_preds))
else:
    meta_preds = []
    print("No Meta cache found, starting from 0")

texts = df["english"].tolist()
batch_size = 32 if device == "cuda" else 8

for i in range(len(meta_preds), len(texts), batch_size):
    if i % 512 == 0:
        print(f"Meta resume: {i}/{len(texts)}")
    meta_preds.extend(translate_meta_batch(texts[i:i + batch_size]))

    # Save every ~1024 rows
    if len(meta_preds) % 1024 < batch_size:
        pd.DataFrame({"pred": meta_preds}).to_csv(META_FILE, index=False, encoding="utf-8-sig")

pd.DataFrame({"pred": meta_preds}).to_csv(META_FILE, index=False, encoding="utf-8-sig")
df["urdu_meta"] = meta_preds
print("Meta done:", len(meta_preds))

# Preview
df[["english", "urdu_ref", "urdu_google", "urdu_meta"]].head(10)

No Google cache found, starting from 0
Google resume: 0/24524
Google resume: 50/24524
Google resume: 100/24524
Google resume: 150/24524
Google resume: 200/24524
Google resume: 250/24524
Google resume: 300/24524
Google resume: 350/24524
Google resume: 400/24524
Google resume: 450/24524
Google resume: 500/24524
Google resume: 550/24524
Google resume: 600/24524
Google resume: 650/24524
Google resume: 700/24524
Google resume: 750/24524
Google resume: 800/24524
Google resume: 850/24524
Google resume: 900/24524
Google resume: 950/24524
Google resume: 1000/24524
Google resume: 1050/24524
Google resume: 1100/24524
Google resume: 1150/24524
Google resume: 1200/24524
Google resume: 1250/24524
Google resume: 1300/24524
Google resume: 1350/24524
Google resume: 1400/24524
Google resume: 1450/24524
Google resume: 1500/24524
Google resume: 1550/24524
Google resume: 1600/24524
Google resume: 1650/24524
Google resume: 1700/24524
Google resume: 1750/24524
Google resume: 1800/24524
Google resume: 1850/24

,english,urdu_ref,urdu_google,urdu_meta
0,is zain your nephew,زین تمہارا بھتیجا ہے۔,زین تمہارا بھتیجا ہے۔,تمہاری بہن بہن ہے
1,i wish youd trust me,کاش تم مجھ پر بھروسہ کرتے,کاش تم مجھ پر بھروسہ کرتے,میں چاہتا ہوں کہ نوجوان مجھے بھروسہ کریں
2,did he touch you,کیا اس نے آپ کو چھوا؟,کیا اس نے آپ کو چھوا؟,وہ تمہیں لمس کرتی ہے
3,its part of life,اس کی زندگی کا حصہ,اس کی زندگی کا حصہ,زندگی کا حصہ
4,zain isnt ugly,زین بدصورت نہیں ہے۔,زین بدصورت نہیں ہے۔,زینب برا نہیں ہے
5,above all be patient,سب سے بڑھ کر صبر کرو,سب سے بڑھ کر صبر کرو,سب سے پہلے صبر کرنا
6,i learned it from him,میں نے اسے اس سے سیکھا۔,میں نے اسے اس سے سیکھا۔,میں نے اس سے سیکھا
7,why am i doing this,میں یہ کیوں کر رہا ہوں,میں یہ کیوں کر رہا ہوں,میں یہ کیوں کرتا ہوں
8,i made a bad decision,میں نے ایک برا فیصلہ کیا,میں نے ایک برا فیصلہ کیا,میں نے ایک برا فیصلہ کیا
9,zain wont care,زین پرواہ نہیں کرے گا,زین پرواہ نہیں کرے گا,زینب کی دیکھ بھال


**Quick check**

In [15]:
print("Rows:", len(df))
print("Missing urdu_google:", df["urdu_google"].isna().sum())
print("Missing urdu_meta  :", df["urdu_meta"].isna().sum())
df[["english", "urdu_ref", "urdu_google", "urdu_meta"]].head(5)

Rows: 24524
Missing urdu_google: 0
Missing urdu_meta  : 0


,english,urdu_ref,urdu_google,urdu_meta
0,is zain your nephew,زین تمہارا بھتیجا ہے۔,زین تمہارا بھتیجا ہے۔,تمہاری بہن بہن ہے
1,i wish youd trust me,کاش تم مجھ پر بھروسہ کرتے,کاش تم مجھ پر بھروسہ کرتے,میں چاہتا ہوں کہ نوجوان مجھے بھروسہ کریں
2,did he touch you,کیا اس نے آپ کو چھوا؟,کیا اس نے آپ کو چھوا؟,وہ تمہیں لمس کرتی ہے
3,its part of life,اس کی زندگی کا حصہ,اس کی زندگی کا حصہ,زندگی کا حصہ
4,zain isnt ugly,زین بدصورت نہیں ہے۔,زین بدصورت نہیں ہے۔,زینب برا نہیں ہے


**BLEU per sentence + Average BLEU**

In [18]:
import sacrebleu

def sentence_bleu(pred: str, ref: str) -> float:
    return float(sacrebleu.sentence_bleu(str(pred), [str(ref)]).score)

df["bleu_google"] = [sentence_bleu(p, r) for p, r in zip(df["urdu_google"], df["urdu_ref"])]
df["bleu_meta"]   = [sentence_bleu(p, r) for p, r in zip(df["urdu_meta"], df["urdu_ref"])]

avg_bleu_google = float(df["bleu_google"].mean())
avg_bleu_meta   = float(df["bleu_meta"].mean())

print("Average BLEU (Google):", round(avg_bleu_google, 4))
print("Average BLEU (Meta)  :", round(avg_bleu_meta, 4))

Average BLEU (Google): 13.8202
Average BLEU (Meta)  : 6.2135


**ROUGE**

In [19]:
!pip -q install rouge_score
print("rouge_score installed")

  Preparing metadata (setup.py) ... done
rouge_score installed


In [31]:
import re
from collections import Counter
import numpy as np

def normalize_ur(s: str) -> str:
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokens(s: str):
    return normalize_ur(s).split()

def ngrams(toks, n: int):
    return [tuple(toks[i:i+n]) for i in range(0, max(0, len(toks)-n+1))]

def f1(p: float, r: float) -> float:
    return 0.0 if (p + r) == 0 else (2 * p * r) / (p + r)

def rouge_n_f1(pred: str, ref: str, n: int) -> float:
    pt = tokens(pred)
    rt = tokens(ref)
    if len(pt) == 0 or len(rt) == 0:
        return 0.0

    p_ng = Counter(ngrams(pt, n))
    r_ng = Counter(ngrams(rt, n))
    overlap = sum((p_ng & r_ng).values())

    p_total = sum(p_ng.values())
    r_total = sum(r_ng.values())
    if p_total == 0 or r_total == 0:
        return 0.0

    prec = overlap / p_total
    rec = overlap / r_total
    return f1(prec, rec)

def lcs_length(a, b) -> int:
    # LCS length for token lists (O(n*m) DP). Works fine for short sentences.
    n, m = len(a), len(b)
    dp = [0] * (m + 1)
    for i in range(1, n + 1):
        prev = 0
        for j in range(1, m + 1):
            tmp = dp[j]
            if a[i-1] == b[j-1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j-1])
            prev = tmp
    return dp[m]

def rouge_l_f1(pred: str, ref: str) -> float:
    pt = tokens(pred)
    rt = tokens(ref)
    if len(pt) == 0 or len(rt) == 0:
        return 0.0
    lcs = lcs_length(pt, rt)
    prec = lcs / len(pt)
    rec = lcs / len(rt)
    return f1(prec, rec)

def rouge_scores_avg(preds, refs):
    r1, r2, rL = [], [], []
    for p, r in zip(preds, refs):
        r1.append(rouge_n_f1(p, r, 1))
        r2.append(rouge_n_f1(p, r, 2))
        rL.append(rouge_l_f1(p, r))
    return {
        "rouge1": float(np.mean(r1)),
        "rouge2": float(np.mean(r2)),
        "rougeL": float(np.mean(rL)),
    }

rouge_google = rouge_scores_avg(df["urdu_google"].tolist(), df["urdu_ref"].tolist())
rouge_meta   = rouge_scores_avg(df["urdu_meta"].tolist(), df["urdu_ref"].tolist())

print("Urdu-friendly ROUGE (Google):", rouge_google)
print("Urdu-friendly ROUGE (Meta)  :", rouge_meta)

Urdu-friendly ROUGE (Google): {'rouge1': 0.16567121859340872, 'rouge2': 0.12070000631391872, 'rougeL': 0.16525669248996688}
Urdu-friendly ROUGE (Meta)  : {'rouge1': 0.11286330086917917, 'rouge2': 0.041723676610179866, 'rougeL': 0.11149154668577746}


**Final comments (which is better?)**

In [33]:
def winner(a: float, b: float) -> str:
    if abs(a - b) < 1e-9:
        return "Tie"
    return "Google" if a > b else "Meta (Facebook/M2M100)"

print("BLEU winner   :", winner(avg_bleu_google, avg_bleu_meta))
print("ROUGE-L winner:", winner(rouge_google["rougeL"], rouge_meta["rougeL"]))

print(f"- Average BLEU shows {winner(avg_bleu_google, avg_bleu_meta)} performs better on this Eng→Ur dataset.")
print(f"- Urdu-friendly ROUGE-L shows {winner(rouge_google['rougeL'], rouge_meta['rougeL'])} performs better in overlap/recall.")

BLEU winner   : Google
ROUGE-L winner: Google
- Average BLEU shows Google performs better on this Eng→Ur dataset.
- Urdu-friendly ROUGE-L shows Google performs better in overlap/recall.


**Save results**

In [34]:
import os
import json
from google.colab import files

os.makedirs("outputs", exist_ok=True)

out_csv = "outputs/final_translation_results.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")

summary = {
    "num_samples": len(df),
    "avg_bleu_google": float(avg_bleu_google),
    "avg_bleu_meta": float(avg_bleu_meta),

    # clearer keys (Urdu-friendly ROUGE)
    "urdu_friendly_rouge_google": rouge_google,
    "urdu_friendly_rouge_meta": rouge_meta,
}

with open("outputs/final_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved:", out_csv, "and outputs/final_summary.json")

files.download(out_csv)
files.download("outputs/final_summary.json")

Saved: outputs/final_translation_results.csv and outputs/final_summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>